# 05 — RQ2: Does Temperature Explain Electricity Demand?

**Question:** Does temperature explain Austrian electricity demand — and does that
relationship change with the seasons?

Austria's demand peaks in winter and dips in summer (EDA result), and the temperature
curve is its near-perfect mirror. This notebook asks how much of that swing temperature
actually accounts for, and pins down the *shape* of the relationship rather than
assuming a straight line. Where RQ1 traced *what* Austria generates, RQ2 is the
demand-side weather story: *why load moves*.

## Data

Two hourly tables, both stored UTC, joined 1:1 on `ts_utc`:

| Table | Grain | Source | Column used |
|---|---|---|---|
| `demand`  | hourly | ENTSO-E (APG), resampled from 15-min | `demand_mw` |
| `weather` | hourly | Open-Meteo / ERA5, Vienna | `temperature_c` |

The orphan extra hour is in `prices` only, so `demand ⋈ weather` is clean — six full
years, 2019–2024. All clock-dependent aggregation converts to `Europe/Vienna` at the
display layer.

## Method — OLS + STL, and the confounds we already know

From EDA we expect a **nonlinear, asymmetric** relationship, not a clean line:

- **Heating-dominated.** Demand climbs steeply as it gets cold, but barely responds to
  heat — little summer cooling load (minimal AC in Austria). The cold-side slope ≫ the
  warm-side slope.
- **Strong calendar structure.** Demand has a pronounced hour-of-day cycle (morning +
  evening peaks) and a weekday > weekend gap. These move demand *independently* of
  temperature, so they must be controlled for or aggregated out — otherwise they bias
  the temperature term and inflate the residual.

**OLS** (ordinary least squares) quantifies the relationship — *how many MW per degree*,
once the calendar confounds are handled. The functional form (a degree-day transform vs
a piecewise / spline temperature term) and the aggregation grain (daily vs
hourly-with-controls) are **decided from the Step 1 eyeball**, not assumed here.

**STL** (Seasonal-Trend decomposition using Loess) adds the time-series view: it splits
demand into *trend + seasonal cycle + remainder*. That isolates the seasonal swing the
temperature model is trying to explain, exposes any slow trend (efficiency gains,
post-COVID demand), and leaves a clean remainder to sanity-check the regression against.

| Step | What | Output |
|---|---|---|
| 1 | Join, confirm the window, eyeball the temp–demand shape at daily grain | Bivariate scatter |
| 2 | Spec the regression (grain, temp functional form, confound handling) + frame STL | — (decided from Step 1) |
| 3 | Fit OLS + run STL; headline visual + 2-sentence finding | Model + decomposition plots |

**Note — scope honesty.** We explain *national* demand with *Vienna* temperature: ERA5
at a single location, as a proxy for the country's population-weighted climate. It's a
deliberate and defensible simplification (Vienna is the demand centre), but worth
stating outright — one station isn't the whole country.

## Setup

In [ ]:
# imports, paths, house style, read-only DuckDB connection
import sys
from pathlib import Path
from functools import partial
from typing import cast

import duckdb
import holidays  # if needed: conda install -c conda-forge holidays
import matplotlib.pyplot as plt
from IPython.display import display
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.tsa.seasonal import STL

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.viz import set_house_style, PALETTE, save_headline_fig, save_qa_fig

set_house_style()
save_qa = partial(save_qa_fig, notebook="05_rq2_temperature_demand")

# read_only=True — RQ notebooks only SELECT; a stray write raises instead of mutating the DB
DB_PATH = PROJECT_ROOT / "data" / "processed" / "austria_energy.duckdb"
con = duckdb.connect(str(DB_PATH), read_only=True)


## 1 · Prove the 1:1 join & confirm the window

In [ ]:
# if demand, weather, and the join all report the same n / lo / hi, the join is
# exactly one-to-one: the prices-only orphan hour didn't leak in, and no
# demand/weather hour got dropped. (USING(ts_utc) = join on the equally-named key.)
con.sql("""
SELECT 'demand'         AS tbl, COUNT(*) AS n, MIN(ts_utc) AS lo, MAX(ts_utc) AS hi
FROM demand
UNION ALL
SELECT 'weather',             COUNT(*),       MIN(ts_utc),       MAX(ts_utc)
FROM weather
UNION ALL
SELECT 'demand JOIN weather', COUNT(*),       MIN(ts_utc),       MAX(ts_utc)
FROM demand JOIN weather USING (ts_utc)
""").df()

## 2 · Daily means & the temp–demand scatter

In [ ]:
# grain = daily (local). Hourly would smear the relationship under the hour-of-day
# cycle; the daily mean collapses that cycle and surfaces the heating signal.
daily = con.sql("""
WITH joined AS (
    SELECT
        (d.ts_utc AT TIME ZONE 'Europe/Vienna')  AS ts_local,   -- convert once
        d.demand_mw,
        w.temperature_c
    FROM demand d
    JOIN weather w USING (ts_utc)        -- 1:1 inner join (verified in Section 1)
)
SELECT
    CAST(ts_local AS DATE)  AS local_date,
    AVG(temperature_c)      AS temp_c,
    AVG(demand_mw)          AS demand_mw,
    COUNT(*)                AS n_hours    -- 24 normally; 23/25 on DST days
FROM joined
WHERE ts_local < TIMESTAMP '2025-01-01'  -- last UTC hour spills to 2025 local → drop 1-h tail day
GROUP BY local_date
ORDER BY local_date
""").df()

# season tag (display-layer only; drives colour, shows the heating asymmetry)
season_of = {12: "Winter", 1: "Winter", 2: "Winter",
             3: "Spring", 4: "Spring", 5: "Spring",
             6: "Summer", 7: "Summer", 8: "Summer",
             9: "Autumn", 10: "Autumn", 11: "Autumn"}
daily["season"] = pd.to_datetime(daily["local_date"]).dt.month.map(season_of)
season_colors = {"Winter": PALETTE["hydro"], "Spring": PALETTE["wind"],
                 "Summer": PALETTE["temp"],  "Autumn": PALETTE["solar"]}

# guide-to-the-eye only (not a model fit): mean demand per 2 °C temp bin
edges  = np.arange(cast(float, np.floor(daily["temp_c"].min())),
                   cast(float, np.ceil(daily["temp_c"].max())) + 2, 2)
binned = (cast(pd.Series,
               daily.assign(tbin=pd.cut(daily["temp_c"], edges, include_lowest=True))
                    .groupby("tbin", observed=True)["demand_mw"].mean())
          .reset_index())
binned["temp_mid"] = binned["tbin"].apply(lambda iv: iv.mid)

fig, ax = plt.subplots()
for s in ["Winter", "Spring", "Summer", "Autumn"]:
    sub = daily[daily["season"] == s]
    ax.scatter(sub["temp_c"], sub["demand_mw"], s=12, alpha=0.5,
               color=season_colors[s], label=s)
ax.plot(binned["temp_mid"], binned["demand_mw"], color="#1A1A1A", lw=2,
        marker="o", ms=4, label="2 °C binned mean")
ax.set(title="Austria — daily mean demand vs temperature, 2019–2024",
       xlabel="daily mean temperature (°C)", ylabel="daily mean demand (MW)")
ax.legend(ncol=5, fontsize=9, loc="upper right")
plt.tight_layout()
save_qa(fig, "austria_daily_mean_demand_vs_temperature")
plt.show()

display(daily["n_hours"].value_counts().sort_index())  # sanity: mostly 24

## 3 · Build the daily modelling frame

In [ ]:
# daily demand + temperature (from the demand⋈weather join) + the two calendar
# controls. The heating/cooling degree-days (HDD/CDD) are not built here — they
# depend on the base temperature, which we estimate by grid search in Section 4.
model_df = con.sql("""
WITH joined AS (
    SELECT
        (d.ts_utc AT TIME ZONE 'Europe/Vienna')  AS ts_local,
        d.demand_mw,
        w.temperature_c
    FROM demand d
    JOIN weather w USING (ts_utc)
)
SELECT
    CAST(ts_local AS DATE)  AS local_date,
    AVG(temperature_c)      AS temp_c,
    AVG(demand_mw)          AS demand_mw
FROM joined
WHERE ts_local < TIMESTAMP '2025-01-01'   -- drop the 1-hour 2025 tail day
GROUP BY local_date
ORDER BY local_date
""").df()

# calendar controls (feature layer, pandas)
model_df["local_date"] = pd.to_datetime(model_df["local_date"])

# weekend: Saturday=5, Sunday=6 (Monday=0)
model_df["is_weekend"] = (model_df["local_date"].dt.dayofweek >= 5).astype(int)

# Austrian national public holidays — they depress demand much like weekends
at_holidays = holidays.Austria(years=range(2019, 2025))
model_df["is_holiday"] = (
    model_df["local_date"].dt.date.map(lambda d: d in at_holidays).astype(int)
)

# sanity check before we model on it
print(f"rows:          {len(model_df)}")
print(f"weekend days:  {model_df['is_weekend'].sum():>4}   (~2/7 of rows ≈ 626)")
print(f"holiday days:  {model_df['is_holiday'].sum():>4}   (~13/yr × 6 yr ≈ 78)")
print(f"temp range:    {model_df['temp_c'].min():.1f} … {model_df['temp_c'].max():.1f} °C")
model_df.head()

## 4 · Balance temperature & degree-day regression

In [ ]:
# estimate the balance temperature, then fit the degree-day regression with
# heteroskedasticity- and autocorrelation-consistent (HAC, Newey–West) standard errors.

# grid-search the balance temperature T_base
# Build HDD/CDD at each candidate base, fit, keep the base with the best R².
# R² is unaffected by the SE method, so we search with ordinary errors here,
# then refit once with HAC below.
def fit_at_base(df: pd.DataFrame, base: float):
    """Fit the degree-day OLS at a given base temperature; return the result."""
    d = df.assign(
        hdd=np.maximum(0.0, base - df["temp_c"]),  # heating degree-days
        cdd=np.maximum(0.0, df["temp_c"] - base),  # cooling degree-days
    )
    return smf.ols("demand_mw ~ hdd + cdd + is_weekend + is_holiday", data=d).fit()

bases  = np.arange(10.0, 22.01, 0.5)
scores = [(b, fit_at_base(model_df, b).rsquared) for b in bases]
t_base = max(scores, key=lambda bx: bx[1])[0]
print(f"estimated balance temperature T_base = {t_base:.1f} °C")

# final model at T_base, with HAC / Newey–West standard errors.
# maxlags ≈ days of autocorrelation to absorb (rule-of-thumb ≈ 8; 10 is a touch
# conservative — about a week and a half of weather persistence).
model_df = model_df.assign(
    hdd=np.maximum(0.0, t_base - model_df["temp_c"]),
    cdd=np.maximum(0.0, model_df["temp_c"] - t_base),
)
fit = smf.ols(
    "demand_mw ~ hdd + cdd + is_weekend + is_holiday", data=model_df
).fit(cov_type="HAC", cov_kwds={"maxlags": 10})

print(fit.summary())

## 5 · STL decomposition & seasonal check

In [ ]:
# STL splits daily demand into trend + seasonal (annual) + remainder, from the
# demand series alone — no temperature. period=365 = annual cycle; robust=True
# downweights outliers (holiday spikes) so they don't bend the trend.

# regular daily series, gap-free (2192 consecutive days → asfreq adds no NaN)
s = model_df.set_index("local_date")["demand_mw"].asfreq("D")
assert cast(bool, s.notna().all()), "unexpected gap in the daily series"

res = STL(s, period=365, robust=True).fit()

# the decomposition (observed / trend / seasonal / remainder)
fig = res.plot()
fig.set_size_inches(12, 8)
fig.suptitle("STL decomposition of daily demand (period = 365)", y=1.01)
plt.tight_layout()
plt.show()

# consistency check: seasonal swing vs the temperature cycle.
# Collapse to one representative year (mean by day-of-year), compare shapes.
# STL found this swing without temperature, so agreement is corroboration —
# not proof (point 5). The causal weight stays with the regression.
comp = pd.DataFrame({
    "seasonal": res.seasonal,
    "temp_c":   model_df.set_index("local_date")["temp_c"],
})
comp["doy"] = comp.index.to_series().dt.dayofyear
doy = cast(pd.DataFrame, comp.groupby("doy").mean(numeric_only=True))

fig, ax = plt.subplots()
ax.plot(doy.index, doy["seasonal"], color=PALETTE["demand"], lw=1.8,
        label="STL seasonal component (demand)")
ax.axhline(0, color=PALETTE["muted"], lw=0.8)
ax.set_xlabel("day of year")
ax.set_ylabel("demand seasonal component (MW)")

ax2 = ax.twinx()
ax2.plot(doy.index, doy["temp_c"], color=PALETTE["temp"], lw=1.8,
         label="mean temperature")
ax2.set_ylabel("mean temperature (°C)")
ax2.grid(False)

ax.set_title("Seasonal demand swing vs the temperature cycle — mirror images")
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="upper center", ncol=2, fontsize=9)
plt.tight_layout()
save_qa(fig, "austria_seasonal_demand_swing_vs_temperature_cycle")
plt.show()

## 6 · Headline — the degree-day fit

In [ ]:
# headline figure + finding
# The degree-day fit over the data. Each day's demand is adjusted to a weekday
# baseline (fitted weekend/holiday offsets removed) so the cloud centres on the
# temperature response and the kinked curve fits cleanly.
b = fit.params

# calendar-adjusted demand (remove weekend + holiday offsets)
adj = (model_df["demand_mw"]
       - b["is_weekend"] * model_df["is_weekend"]
       - b["is_holiday"] * model_df["is_holiday"])

# fitted temperature curve (weekday baseline) on a smooth grid
tgrid = np.linspace(model_df["temp_c"].min(), model_df["temp_c"].max(), 200)
hdd_g = np.maximum(0.0, t_base - tgrid)
cdd_g = np.maximum(0.0, tgrid - t_base)
curve = b["Intercept"] + b["hdd"] * hdd_g + b["cdd"] * cdd_g

season_of = {12:"Winter",1:"Winter",2:"Winter",3:"Spring",4:"Spring",5:"Spring",
             6:"Summer",7:"Summer",8:"Summer",9:"Autumn",10:"Autumn",11:"Autumn"}
season = model_df["local_date"].dt.month.map(season_of)
colors = {"Winter":PALETTE["hydro"], "Spring":PALETTE["wind"],
          "Summer":PALETTE["temp"],  "Autumn":PALETTE["solar"]}

fig, ax = plt.subplots(figsize=(11, 6))
for s in ["Winter","Spring","Summer","Autumn"]:
    m = season == s
    ax.scatter(model_df.loc[m, "temp_c"], adj[m], s=12, alpha=0.45,
               color=colors[s], label=s)
ax.plot(tgrid, curve, color="#1A1A1A", lw=2.4, label="fitted degree-day model")
ax.axvline(t_base, color=PALETTE["muted"], ls="--", lw=1)
ax.annotate(f"balance point {t_base:.1f} °C", (t_base, ax.get_ylim()[0]),
            xytext=(6, 8), textcoords="offset points", fontsize=9,
            color=PALETTE["demand"])

txt = (f"heating  +{b['hdd']:.0f} MW/°C\n"
       f"cooling  +{b['cdd']:.0f} MW/°C\n"
       f"R² = {fit.rsquared:.2f}")
ax.text(0.97, 0.95, txt, transform=ax.transAxes, ha="right", va="top", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec=PALETTE["muted"], lw=0.8))

ax.set_xlabel("daily mean temperature (°C)")
ax.set_ylabel("daily mean demand (MW), weekend/holiday effect removed")
ax.set_title("Austria — temperature drives daily electricity demand, asymmetrically")
ax.legend(loc="upper center", ncol=5, fontsize=9, bbox_to_anchor=(0.5, -0.08))
plt.tight_layout()
save_headline_fig(fig, "rq2_temperature_demand")
plt.show()

In [ ]:
# close the connection
con.close()

## Finding

**Finding (RQ2).** Temperature is the dominant driver of Austria's daily electricity
demand, and the relationship is strongly asymmetric: below a 16.5 °C balance point each
degree of cooling adds ≈105 MW (heating), while above it each degree adds only ≈16 MW —
a weak, marginally significant cooling response, reflecting Austria's minimal
air-conditioning — with temperature plus weekend/holiday effects explaining 79% of daily
variance under robust (HAC) standard errors. An independent STL decomposition confirms
the seasonal demand swing is the mirror image of the temperature cycle, yet also exposes
a multi-year baseline decline from 2022 (energy-crisis demand destruction) that
temperature does *not* explain — the weather model captures the seasonal and day-to-day
swing, not the structural trend.